## Lab Assignment - 3

#### Fine-tune GPT or GPT-2 for creative story generation.


In [1]:
# install required libraries
!pip install transformers datasets accelerate -q

In [2]:
# import libraries
import pandas as pd
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

In [3]:
# creating a small story dataset
stories = [
    "Once upon a time in a magical forest, a small rabbit discovered a glowing stone that could talk.",
    "In a distant galaxy, a lonely robot learned the meaning of friendship when a child visited his planet.",
    "A young girl found a secret door in her grandmother's house that led to a world of dragons.",
    "On a rainy evening, a detective received a mysterious letter that changed his life forever.",
    "Deep under the ocean, a curious dolphin discovered an ancient underwater city."
]

df = pd.DataFrame(stories, columns=["text"])

dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['text'],
    num_rows: 5
})

In [4]:
# Load GPT-2 Model and Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 has no pad token, so set it
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# Tokenize the Dataset
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [6]:
# Data Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [8]:
# Training Configuration
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-story",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2
)

In [9]:
# tarier step
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [10]:
# tarin the model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=2.5478441874186197, metrics={'train_runtime': 148.2597, 'train_samples_per_second': 0.169, 'train_steps_per_second': 0.101, 'total_flos': 1633075200000.0, 'train_loss': 2.5478441874186197, 'epoch': 5.0})

In [11]:
# generate a story
prompt = "Once upon a time"

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs["input_ids"],
    max_length=100,
    num_return_sequences=1,
    temperature=0.9,
    top_k=50
)

generated_story = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_story)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Once upon a time, a young girl discovered a secret that could change the world forever.

The world was a dark place, filled with evil.


But when a mysterious power came to the world, the world changed.

The world was a dark place, filled with evil.

But when a mysterious power came to the world, the world changed.

The world was a dark place, filled with evil.

But when a mysterious power came to the


In [12]:
# saving the model
model.save_pretrained("story_generator_model")
tokenizer.save_pretrained("story_generator_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('story_generator_model/tokenizer_config.json',
 'story_generator_model/tokenizer.json')